In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import pytest
from ucimlrepo import fetch_ucirepo 


# Coisas a serem feitas:

- [ ] Validar se: A implementação deve ser generalizada para qualquer quantidade de atributos.
- [ ] **Testes Unitários:** validar individualmente métodos como `fit()`, `predict()`, funções de erro, entre outros.
- [ ] **Testes Funcionais:** validar o pipeline completo de treinamento e predição com conjuntos de dados simples.

---
#### Calibração da Taxa de Aprendizado:
* Utilizar o conjunto de desenvolvimento para encontrar a melhor taxa de aprendizado (*learning rate*).
* Após encontrar o melhor valor, retreinar o modelo utilizando o conjunto de **treinamento + desenvolvimento**, e avaliar no conjunto de **teste**.

#### Análises Obrigatórias:
* **Curva de Treinamento:**
  * Gerar um gráfico que mostre a evolução da função de erro durante o treinamento.
  * Realizar uma análise textual sobre o comportamento da curva de erro.
* **Análise dos Parâmetros:**
  * Criar um gráfico de barras exibindo os coeficientes encontrados.
  * Analisar em texto quais parâmetros são mais relevantes positiva e negativamente para a predição.

#### Avaliação Quantitativa:
* Calcular e reportar as métricas:
  * **MAE** (*Mean Absolute Error*)
  * **MSE** (*Mean Squared Error*)
  * **MAPE** (*Mean Absolute Percentage Error*)
* As métricas devem ser implementadas em módulos ou classes próprias.
* Comparar os erros de treinamento+desenvolvimento e teste.
* Analisar se há *overfitting* ou *underfitting*, justificando os resultados.

#### Análise de Tempo de Treinamento:
* Medir o tempo total de treinamento.
* Comentar sobre o desempenho temporal observado.

In [2]:
# Gerando os dados sintéticos, função y = x^2

df_sintetico = pd.DataFrame({
                'x': range(50),
                'y': [i*2 for i in range(50)]
})

In [3]:
df_sintetico.head(5)

,x,y
0,0,0
1,1,2
2,2,4
3,3,6
4,4,8


# Fórmula dos gradiente dos mínimos quadradados para múltiplos atributos:

- Função de custo
$$J(\theta) = \frac{1}{2m} \sum_{i=1}^m (h_\theta(x^{(i)}) - y_i)^2$$

- Gradiente descedente para múltiplos atributos:

$$\theta _j := \theta _j - \alpha \frac{\partial}{\partial \theta _j} J(\theta)$$

# Como funciona

$\theta ^T$ = $\begin{bmatrix}\theta _0 & \theta _1 & \theta_2 & ... & \theta _n \end{bmatrix}_{1xn}$ 

x = $\begin{bmatrix} x_0\\ x_1  \\ ...  \\ x_n \end{bmatrix}_{nx1}$

Ou seja, cada linha corresponde a uma feature.

Onde a quantidade de colunas de j determinarão a quantidade de "predições" feitas pelo modelo, dado que a multiplicação de $\theta ^T$ * x vai ser dada por:

$ \theta ^T * x $ = $ \theta _0 * x_0 + \theta _1 * x_1 + ... + \theta _n * x_n $

- Com as dimensões de x sendo $x_{nx2}$, teríamos algo como:

$ \theta ^T * x $ = $\begin{bmatrix} \theta _0 * x_0^0 + \theta _1 * x_1^0 + ... + \theta _n * x_n^0 & \theta _0 * x_0^1 + \theta _1 * x_1^1 + ... + \theta _n * x_n^1  \end{bmatrix}_{1x2}$


In [4]:
class EvaluativeMetrics:

    # Instânciando a classe
    def __init__(self, y_pred: np.ndarray, y: np.ndarray):
        self.y_pred = y_pred
        self.y = y

    # Métrica MAE
    def MAE(self) -> float:
        """"""
        return 1/len(self.y_pred) * np.sum(abs(self.y_pred - self.y))

    # Métrica MAPE
    def MAPE(self) -> float:
        """"""
        return 100/len(self.y_pred) * np.sum((self.y_pred - self.y)/self.y)

    # Métrica MSE:
    def MSE(self) -> float:
            """"""
            return 1/len(self.y_pred) * np.sum((self.y_pred - self.y)**2)



In [ ]:
class RTrainer:
    # Etapa de treinamento, é a mais simples, implementação está no slide
    def __init__(self, a, ephocs, erro=0, logger=False):
        """Construtor da classe RTrainer"""
            
        self.a = a # Taxa de aprendizado
        self.ephocs = ephocs # Quantidade de iterações do modelo
        self.erro = erro # Erro mínimo, um parâmetro adicional na hora de iterar, opcional
        self.logger = logger # Um log para acompanharmos algumas coisas que possam ser interessantes de serem observadas
        self.parametros: np.ndarray

    def fit(self, dados_entrada_treinamento: np.ndarray, y_treinamento: np.ndarray, dados_entrada_val: np.ndarray, y_val: np.ndarray ) -> np.ndarray:
        """ Método que realiza o treinamento do modelo """
    
        # Antes de começar o treinamento, vamos primeiro modelar os dados de forma a facilitar o próprio

        x_treinamento = np.append(
                [np.ones((dados_entrada_treinamento).shape[0])], # O shape é para preencher com a quantidade de dados de entrada que temos nas outras features, esse aqui é o x0 -> vetor de 1s
                # TODO: Validar, precisei adicionar '[]' porque só temos uma coluna de entrada
                np.transpose(dados_entrada_treinamento),  # Necessário transpor porque cada linha fica como uma feature
                axis=0 # Garantindo que vamos manter uma estrutura de: cada linha é uma feature, e cada coluna é uma entrada que resulta nos dados de treinamento, que acaba sendo o contrário do que temos no slide
            )

        # Matriz de validação
        x_val = np.append(
                [np.ones((dados_entrada_val).shape[0])], 
                np.transpose(dados_entrada_val), 
                axis=0
            )
            
        # Ok, temos a matriz dos dados de entrada, agora precisamos criar os parâmetros

        self.parametros = np.ones(x_treinamento.shape[0]) # TODO: adicionar um parâmetro opcional que possa definir o tipo de inicialização desse vetor de parâmetros, se é rand, 0 e 1, vai ser um if né simples

        # Agora podemos realizar as predições

        for i in range(self.ephocs+1):

            # Primeiro, precisamos calcular o vetor y_pred, que vai ser uma multiplicação de matrizes

            y_pred = self.parametros @ x_treinamento # O "@" é o operador de multiplicação de matrizes do numpy
            y_pred_val = self.parametros @ x_val

            #print(y_pred)
            #print(y_treinamento)
            # Segundo, calculando a função de custo

            # TODO: substituir isso aqui pelos os dados de validação, porque se o custo for sobre os dados de treinamento, o modelo estará se recompensando por estar "decorando"
            # ele, precisamos recompensar ele apenas se sair bem em um conjunto que ele não "vê"

            custo = 1/(2*len(x_val[0])) * np.sum((y_pred_val - y_val)**2) # Só se quisermos ver o comportamento do custo, mas não é necessário essa linha aqui
            
            if i%1 == 0:
                print(custo)
            # Solução com erro 10-3

            if custo < 0.001:
                # print("quebrado!") -> loggando
                break
            
            parametros_temp = self.parametros.copy()

            # Terceiro, aplicando o método do gradiente descendente para múltiplos atributos

            for j in range(x_treinamento.shape[0]): 
                # Iterando por feature
                parametros_temp[j] = self.parametros[j] - self.a * (1/(len(x_treinamento[0]))) * np.sum((y_pred - y_treinamento) * x_treinamento[j])

            self.parametros = parametros_temp

        return self.parametros


    #  TODO: fazendo retornar um valor só, rever isso aqui
    def predict(self, dados_predicao: np.ndarray) -> float:
        """Método que realiza previsão do modelo, vai retornar -> um vetor de parâmetros OU um objeto MODEL contendo ao menos uma propriedade que indique o número de parâmetros"""

        print(dados_predicao)  # Precisamos adicionar mais um valor
        teste = pd.concat([pd.Series(1), dados_predicao]) # Dessa forma resolvermos a questão do theta0
        print(teste)
        print(self.parametros) # Tem 12 valores, por causa do theta0
        return  self.parametros @ np.transpose(teste)
        

In [37]:
# TODO: Célula de testes


def testando_fit_modelo(modelo: RTrainer, dados_entrada: np.ndarray, y: np.ndarray):
    
    try:
        pesos = modelo.fit(dados_entrada, dados_entrada)
    except Exception as e:
        pytest.fail(f"O fit do modelo falhou por: {e}")

    # Verificação se os pesos que estão saindo do modelo estão tipados como se espera
    assert isinstance(pesos, np.ndarray)

    # Verificação da quantidade de pesos é n+1 da quantidade de colunas de entrada. Onde a quantidade de dimensões corresponde a quantidade de colunas e o shape corresponde a quatidade de parâmetros
    assert (dados_entrada.ndim + 1) == pesos.shape[0]

# TODO: Construir outras validações aqui

In [46]:
testando = RTrainer(
    a = 0.0001,
    ephocs=10000,
    logger=True
)

#testando_fit_modelo(testando, df_sintetico['x'].to_numpy(), df_sintetico['y'].to_numpy() )
#vetores_peso = testando.fit(df_sintetico['x'].to_numpy(), df_sintetico['y'].to_numpy()) # TODO: corrigir isso aqui para poder testar

# Dataset real

### [ X  ] Validar: Dataset Real
* Utilizar um dataset público voltado para regressão disponível no [UCI Machine Learning Repository](https://archive.ics.uci.edu/) ou no [Kaggle](https://www.kaggle.com/).

---

### [ x ] Validar: Divisão dos Dados
* Separar o dataset em três conjuntos: **treinamento**, **desenvolvimento** e **teste**.


In [9]:
# fetch dataset 
wine_quality = fetch_ucirepo(id=186) 
  
# data (as pandas dataframes) 
X = wine_quality.data.features 
y = wine_quality.data.targets 

In [10]:
X.tail()

,fixed_acidity,volatile_acidity,citric_acid,residual_sugar,chlorides,free_sulfur_dioxide,total_sulfur_dioxide,density,pH,sulphates,alcohol
6492,6.2,0.21,0.29,1.6,0.039,24.0,92.0,0.99114,3.27,0.50,11.2
6493,6.6,0.32,0.36,8.0,0.047,57.0,168.0,0.99490,3.15,0.46,9.6
6494,6.5,0.24,0.19,1.2,0.041,30.0,111.0,0.99254,2.99,0.46,9.4
6495,5.5,0.29,0.30,1.1,0.022,20.0,110.0,0.98869,3.34,0.38,12.8
6496,6.0,0.21,0.38,0.8,0.020,22.0,98.0,0.98941,3.26,0.32,11.8


In [11]:
y.tail()

,quality
6492,6
6493,5
6494,6
6495,7
6496,6


In [67]:
def train_test_validation_split(entrada: pd.DataFrame, saida: pd.DataFrame | None = None, output: str | None = None):
    """ Função que vai fazer o embaralhamento dos dados, para garantir que não haja nenhum viés no treinamento. Além disso, separamos os dados em treinamento, validação e teste. Vamos passar um único Dataframe e vamos separar aqui dentro"""

    # TODO: se estiver a fim, da para deixar bem redondinho essa função, mas por enquanto não vou fazer
    # TODO: permitir ajustar o tamanho dos cortes

    if saida is None and output is None:
        # Ver trowexception
        return("Erro: preciso de: ou um dataframe de saída ou de uma coluna referenciando o output no dataframe de entrada")

    coluna_Y = ''
    randomizando_dataframe = ''

    # Se não tiver sido passado uma string, e sim um dataframe, vamos pegar a coluna de saída e juntar no dataframe de entrada
    if output is None:
        coluna_Y = saida.columns[0]
        randomizando_dataframe = pd.concat([entrada, saida], axis=1)
    else:
        coluna_Y = output
        randomizando_dataframe = entrada.copy()

    # Processo de "bagunçar" o dataframe
    randomizando_dataframe = randomizando_dataframe.iloc[np.random.permutation(len(randomizando_dataframe))].reset_index(drop=True) # -> Usando o numpy para rearranjar os elementos, mantendo os índices normais

    # TODO: caso for ajustar os valores do corte, é só ajustar nesses valores float aqui
    corteTeste = int(len(randomizando_dataframe)*0.70)
    corteValidacao = int(len(randomizando_dataframe)*0.90)

    parteTreinamento = randomizando_dataframe[:corteTeste] # Primeiros 70% do dataframe
    parteValidacao = randomizando_dataframe[corteTeste:corteValidacao] # Próximos 20%
    parteTeste = randomizando_dataframe[corteValidacao:] # Últimos 10%

    # Separando os três dataframes:

    colunas_X = []

    for i in randomizando_dataframe.columns:
        if i != coluna_Y:
            colunas_X.append(i)

    X_train, y_train = parteTreinamento[colunas_X].copy(), parteTreinamento[coluna_Y].copy()
    X_val, y_val = parteValidacao[colunas_X].copy(), parteValidacao[coluna_Y].copy()
    X_test, y_test = parteTeste[colunas_X].copy(),  parteTeste[coluna_Y].copy()

    return X_train, y_train, X_val, y_val, X_test, y_test


In [68]:
X_train, y_train, X_val, y_val, X_test, y_test = train_test_validation_split(X, y)

In [69]:
X_train

,fixed_acidity,volatile_acidity,citric_acid,residual_sugar,chlorides,free_sulfur_dioxide,total_sulfur_dioxide,density,pH,sulphates,alcohol
0,8.6,0.22,0.36,1.90,0.064,53.0,77.0,0.99604,3.47,0.87,11.0
1,7.5,0.26,0.26,18.35,0.084,33.0,139.0,1.00110,3.17,0.39,8.8
2,6.1,0.36,0.41,19.35,0.070,67.0,207.0,1.00118,3.39,0.53,9.1
3,7.4,0.16,0.27,15.50,0.050,25.0,135.0,0.99840,2.90,0.43,8.7
4,5.9,0.26,0.25,12.50,0.034,38.0,152.0,0.99770,3.33,0.43,9.4
...,...,...,...,...,...,...,...,...,...,...,...
4542,6.4,0.64,0.21,1.80,0.081,14.0,31.0,0.99689,3.59,0.66,9.8
4543,5.6,0.12,0.33,2.90,0.044,21.0,73.0,0.98896,3.17,0.32,12.9
4544,6.6,0.30,0.24,1.20,0.034,17.0,121.0,0.99330,3.13,0.36,9.2
4545,6.5,0.40,0.10,2.00,0.076,30.0,47.0,0.99554,3.36,0.48,9.4


In [70]:
X_val

,fixed_acidity,volatile_acidity,citric_acid,residual_sugar,chlorides,free_sulfur_dioxide,total_sulfur_dioxide,density,pH,sulphates,alcohol
4547,7.1,0.28,0.49,6.5,0.041,28.0,111.0,0.992600,3.41,0.58,12.2
4548,7.5,0.24,0.31,13.0,0.049,46.0,217.0,0.998500,3.08,0.53,8.8
4549,5.9,0.23,0.24,3.8,0.038,61.0,152.0,0.991390,3.31,0.50,11.3
4550,6.4,0.46,0.22,14.7,0.047,51.0,183.0,0.998275,3.39,0.60,10.5
4551,6.8,0.51,0.30,4.2,0.066,38.0,165.0,0.994500,3.20,0.42,9.1
...,...,...,...,...,...,...,...,...,...,...,...
5842,6.7,0.17,0.27,1.4,0.032,39.0,149.0,0.992540,3.40,0.52,10.5
5843,6.7,0.19,0.34,1.0,0.022,22.0,94.0,0.991200,3.23,0.57,11.1
5844,7.1,0.71,0.00,1.9,0.080,14.0,35.0,0.997200,3.47,0.55,9.4
5845,7.6,0.19,0.42,1.5,0.044,6.0,114.0,0.991400,3.04,0.74,12.8


In [71]:
X_test

,fixed_acidity,volatile_acidity,citric_acid,residual_sugar,chlorides,free_sulfur_dioxide,total_sulfur_dioxide,density,pH,sulphates,alcohol
5847,8.0,0.250,0.13,17.2,0.036,49.0,219.0,0.99960,2.96,0.46,9.7
5848,7.8,0.965,0.60,65.8,0.074,8.0,160.0,1.03898,3.39,0.69,11.7
5849,6.7,0.130,0.29,5.3,0.051,31.0,122.0,0.99440,3.44,0.37,9.7
5850,7.6,0.480,0.31,9.4,0.046,6.0,194.0,0.99714,3.07,0.61,9.4
5851,6.3,0.240,0.22,11.9,0.050,65.0,179.0,0.99659,3.06,0.58,9.3
...,...,...,...,...,...,...,...,...,...,...,...
6492,6.0,0.280,0.27,15.5,0.036,31.0,134.0,0.99408,3.19,0.44,13.0
6493,7.0,0.300,0.27,1.5,0.076,24.0,145.0,0.99344,3.10,0.52,10.1
6494,6.3,0.130,0.42,1.1,0.043,63.0,146.0,0.99066,3.13,0.72,11.2
6495,8.2,0.600,0.17,2.3,0.072,11.0,73.0,0.99630,3.20,0.45,9.3


In [72]:
y_train

0       7
1       5
2       5
3       7
4       5
       ..
4542    5
4543    8
4544    5
4545    6
4546    8
Name: quality, Length: 4547, dtype: int64

In [73]:
vetores_peso = testando.fit(X_train.to_numpy(), y_train.to_numpy(), X_validation.to_numpy(), y_validation.to_numpy())

AttributeError: 'DataFrame' object has no attribute 'fit'

In [40]:
X_test.iloc[1]

fixed_acidity             7.8000
volatile_acidity          0.2400
citric_acid               0.1800
residual_sugar            6.7000
chlorides                 0.0460
free_sulfur_dioxide      33.0000
total_sulfur_dioxide    160.0000
density                   0.9963
pH                        3.2000
sulphates                 0.5600
alcohol                   9.8000
Name: 5848, dtype: float64

In [48]:
testando.predict(X_test.iloc[1])

fixed_acidity             7.8000
volatile_acidity          0.2400
citric_acid               0.1800
residual_sugar            6.7000
chlorides                 0.0460
free_sulfur_dioxide      33.0000
total_sulfur_dioxide    160.0000
density                   0.9963
pH                        3.2000
sulphates                 0.5600
alcohol                   9.8000
Name: 5848, dtype: float64
0                         1.0000
fixed_acidity             7.8000
volatile_acidity          0.2400
citric_acid               0.1800
residual_sugar            6.7000
chlorides                 0.0460
free_sulfur_dioxide      33.0000
total_sulfur_dioxide    160.0000
density                   0.9963
pH                        3.2000
sulphates                 0.5600
alcohol                   9.8000
dtype: float64
[ 0.88542122 -0.11292621  0.88792645  0.96741064  0.01083864  0.98166706
  0.00775302 -0.00137803  0.88469912  0.61216317  0.90620004  0.1549409 ]


np.float64(5.411214064173185)